<a href="https://colab.research.google.com/github/piaseckazaneta/Python/blob/main/Location_Intelligence.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
!pip install h3 osmnx


import geopandas as gpd
import h3
import matplotlib.pyplot as plt
import osmnx as ox #skrót od OpenStreetMap + NetworkX
from shapely.geometry import Polygon

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 104.7/104.7 kB 4.0 MB/s eta 0:00:00


In [4]:
# 1. Pobieramy obrys miasta
place = "Warszawa, Poland"
city_boundary = ox.geocode_to_gdf(place)


In [6]:
# 2. Konwersja geometrii Shapely na komórki H3 (obsługuje Polygon i MultiPolygon)
geom = city_boundary.geometry.iloc[0]

# Wyciągamy poligony (jeśli miasto ma enklawy / jest MultiPolygonem)
polygons = [geom] if isinstance(geom, Polygon) else list(geom, geoms)

hex_ids = set()
for poly in polygons:
  # Shapely przechowuje (lng, lat) -> H3 v4 wymaga LatLngPoly ze współrzędnymi (lat, lng)
  outer_coords = [(lat, lng) for lng, lat in poly.exterior.coords]
  holes = [[(lat, lng) for lng, lat in hole.coords] for hole in poly.interiors]

  h3_poly = h3.LatLngPoly(outer_coords, holes)
  cells = h3.polygon_to_cells(h3_poly, res=9)
  hex_ids.update(cells)

hex_ids = list(hex_ids)

# 3. Konwersja komórek H3 z powrotem na GeoDataFrame (dla GeoPandas)
hex_polys = [
    Polygon([(lng, lat) for lat, lng in h3.cell_to_boundary(hid)])
    for hid in hex_ids
]
hex_gdf = gpd.GeoDataFrame({'hex_id': hex_ids, 'geometry': hex_polys}, crs="EPSG:4326")

print(f"Sukces! Utworzono {len(hex_gdf)} heksagonów H3 dla miasta {place}.")
hex_gdf.head(3)

Sukces! Utworzono 5331 heksagonów H3 dla miasta Warszawa, Poland.


,hex_id,geometry
0,891f53ca08fffff,"POLYGON ((21.03652 52.33273, 21.03646 52.33108..."
1,891f5234c33ffff,"POLYGON ((20.88294 52.27609, 20.88288 52.27443..."
2,891f5353353ffff,"POLYGON ((21.08873 52.15094, 21.08867 52.14928..."


In [7]:
# 3: Definiujemy tagi OSM, które nas interesują
tags = {
    'amenity': ['cafe', 'restaurant', 'fast_food', 'supermarket'],
    'public_transport': 'platform',
    'highway': 'bus_stop'
}

In [9]:
print("Pobieranie POI z OpenStreetMap (może potrwać nawet 2 min)...")
pois_raw = ox.features_from_place(place, tags)
print(f"Pobrano łącznie {len(pois_raw)}")

Pobieranie POI z OpenStreetMap (może potrwać 10-30 sekund)...
Pobrano łącznie 9260
